[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

*Inference & Decoding*
Implement the accept/reject loop at the heart of **speculative decoding**.

A small draft model proposes `K` tokens; the large target model scores them all
in one batched pass. You then decide how many to keep.

### Signature
```python
def speculative_decode(key, target_probs, draft_probs, draft_tokens):
    ...  # -> list[int]
```

- `target_probs`, `draft_probs`: `(K, V)` probability rows
- `draft_tokens`: `(K,)` proposed token ids
- returns the accepted tokens, plus one resampled token if a rejection happened

### The algorithm
For each position `i`:
1. Accept `draft_tokens[i]` with probability
   $\min\!\left(1, \frac{p_{\text{target}}(t)}{p_{\text{draft}}(t)}\right)$
2. On acceptance, continue to the next position
3. On **rejection**, sample one token from the normalised residual
   $\max(p_{\text{target}} - p_{\text{draft}}, 0)$, append it, and **stop**

If every draft token is accepted, return all `K`.

### Rules
- Guard the division with `1e-10`
- If the residual sums to zero, fall back to a uniform distribution
- Stop at the **first** rejection — do not keep going

### Why this is lossless, not an approximation
This is the property that matters and the one interviewers probe. The
accept/reject rule plus the residual resample is constructed so the emitted
token is distributed **exactly** as $p_{\text{target}}$ — not approximately.

The argument: a token $t$ survives either by being drafted and accepted, with
probability $p_d(t)\min(1, p_t(t)/p_d(t)) = \min(p_d(t), p_t(t))$, or by being
drawn from the residual after a rejection. The two paths sum to exactly
$p_t(t)$. So speculative decoding changes the *speed*, never the output
distribution — which is why you can deploy it without re-evaluating quality.

The speedup comes from the target model scoring `K` positions in **one**
forward pass. Acceptance rate depends on how well the draft mimics the target;
a good pairing keeps 60–80%, giving 2–3× fewer target calls.

### ⚠️ JAX-forced signature change
The original calls `torch.rand` and `torch.multinomial`, drawing from a hidden
global RNG. JAX has none, so the PRNG **key is the first argument**. Split it
per position so the accept coin and the resample draw never reuse randomness.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def speculative_decode(key, target_probs, draft_probs, draft_tokens):
    """Verify drafted tokens against the target distribution.

    Args:
        key:          jax.random key
        target_probs: (K, V) target-model probabilities
        draft_probs:  (K, V) draft-model probabilities
        draft_tokens: (K,) drafted token ids

    Returns:
        list[int] — accepted tokens, plus one resampled token on rejection.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

V = 5
# A draft that agrees with the target is accepted often.
target = jnp.tile(jnp.array([0.6, 0.1, 0.1, 0.1, 0.1]), (4, 1))
draft = jnp.tile(jnp.array([0.5, 0.2, 0.1, 0.1, 0.1]), (4, 1))
tokens = jnp.zeros(4, dtype=jnp.int32)          # all propose token 0

kept = [len(speculative_decode(jax.random.key(s), target, draft, tokens))
        for s in range(200)]
print("mean tokens emitted per round:", sum(kept) / len(kept), "of 4 drafted")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("speculative_decoding")

# hint("speculative_decoding")      # stuck? nudge without the answer
# solution("speculative_decoding")  # spoiler: the reference implementation